# 01 · Volume and adoption

How much the bot was used, by how many of the students who could have used it, where, and how that
changed between last fall and this one.

The one rule this notebook exists to enforce: **the two terms are compared over the same number of
days from their own term start.** Fall 2026 is three weeks old; Fall 2025 ran to mid-December.
Comparing them whole would report the calendar, not a change in behaviour.

In [1]:
# Put the analysis package on the path no matter where Jupyter was started.
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "analysis" / "bloombot_analysis").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "analysis"))

import pandas as pd

from bloombot_analysis import charts, load, metrics, privacy, report, sessions, topics
from bloombot_analysis.config import CONFIG, SURFACE_LABELS, TOPICS

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 40)
print("as of:", CONFIG.as_of)
print("legacy db :", CONFIG.legacy_db, "(exists)" if CONFIG.legacy_db.exists() else "(missing)")
print("current db:", CONFIG.current_db, "(exists)" if CONFIG.current_db.exists() else "(missing)")
print("output    :", CONFIG.out_dir)

as of: 2026-09-25
legacy db : /Users/ab1258/Documents/education_automation/bloombot/tmp/analysis/legacy.db (exists)
current db: /Users/ab1258/Documents/education_automation/bloombot/tmp/analysis/current.db (exists)
output    : /Users/ab1258/Documents/education_automation/bloombot/tmp/analysis/out


In [2]:
sessions_df = pd.read_csv(
    CONFIG.data_path("sessions.csv"), parse_dates=["started_at", "ended_at", "week"]
)
sessions_df["date"] = pd.to_datetime(sessions_df["date"]).dt.date
messages_df = pd.read_csv(CONFIG.data_path("messages.csv"), parse_dates=["ts", "week"])
print(len(sessions_df), "sessions,", len(messages_df), "messages")

259 sessions, 1584 messages


## Adoption — the share of enrolled students who used it at all

The honest headline when volume is small: it has a denominator that means something. Courses with no
enrolment record (everything before Fall 2026 — the old bot had no roster) come back as unmeasurable
rather than as zero.

In [3]:
enrolments_df = pd.read_csv(CONFIG.data_path("enrolments.csv"))
current_term = CONFIG.term(CONFIG.current_term)
this_term_sessions = sessions.in_term(sessions_df, current_term, ts_column="started_at")

adoption_df = sessions.adoption(this_term_sessions, enrolments_df)
adoption_df = adoption_df[["course", "enrolled", "active", "share"]]
adoption_fig = charts.bar_h(
    adoption_df["course"],
    adoption_df["active"],
    "adoption_by_course",
    f"Students who used the bot at least once — {current_term.label} to date",
    "Distinct students",
)
adoption_df

,course,enrolled,active,share
0,Agile Software Development & DevOps,21,4,0.190476
1,Introduction to Programming,46,9,0.195652
2,Software Engineering,34,8,0.235294
3,Web Design,23,3,0.130435


## Weekly activity

Plotted as points, not smoothed and not fitted: with this many weeks a trend line would assert more
than the data supports. The dashed rule marks the start of Fall 2026, when the web and chat-assistant
interfaces launched — a change in *our* product, not in student behaviour.

In [4]:
# `weekly_matrix` zeroes weeks inside a term and leaves the between-term weeks
# empty, so the line breaks over the summer instead of drawing a slope across it.
weekly_wide = sessions.weekly_matrix(sessions_df, by="course")
weekly_fig = charts.line_series(
    weekly_wide,
    "weekly_sessions_by_course",
    "Sessions per week, by course",
    "Sessions",
    annotations=[(current_term.start, "Fall 2026 · web + chat assistant launch")],
)
weekly_wide.tail(8)

course,Agile Software Development & DevOps,Introduction to Programming,Software Engineering,Web Design
week,,,,
2026-08-03,NaN,NaN,NaN,NaN
2026-08-10,NaN,NaN,NaN,NaN
2026-08-17,NaN,NaN,NaN,NaN
2026-08-24,NaN,NaN,NaN,NaN
2026-08-31,2.0,3.0,3.0,2.0
2026-09-07,1.0,5.0,6.0,5.0
2026-09-14,2.0,3.0,1.0,3.0
2026-09-21,1.0,1.0,1.0,2.0


## Where students talked to it

Discord has run for the whole period; web and the chat assistant only exist from Fall 2026. Any
share that mixes them is partly a measure of the launch.

In [5]:
surface_df = sessions.surface_split(this_term_sessions)
surface_df["surface_label"] = surface_df["surface"].map(SURFACE_LABELS).fillna(surface_df["surface"])
surface_fig = charts.bar_h(
    surface_df["surface_label"],
    surface_df["sessions"],
    "sessions_by_surface",
    f"Sessions by interface — {current_term.label} to date",
    "Sessions",
)
surface_df

,surface,sessions,prompts,students,surface_label
0,discord,27,66,17,Discord
1,mcp,8,21,8,Chat assistant
2,web,6,20,5,Web


## Like-for-like: this fall against last fall

Both windows are cut to the same number of days from their own term start.

In [6]:
completeness = sessions.term_completeness()
elapsed = completeness["elapsed_days"]
comparison_term = CONFIG.term(CONFIG.comparison_term)

window_now = sessions.like_for_like(sessions_df, current_term, elapsed, ts_column="started_at")
window_then = sessions.like_for_like(sessions_df, comparison_term, elapsed, ts_column="started_at")

comparison = pd.DataFrame(
    {
        comparison_term.label: [
            len(window_then),
            window_then["person_key"].nunique(),
            int(window_then["prompts"].sum()),
        ],
        current_term.label: [
            len(window_now),
            window_now["person_key"].nunique(),
            int(window_now["prompts"].sum()),
        ],
    },
    index=["Sessions", "Students", "Prompts"],
)
comparison_fig = charts.grouped_bar(
    comparison,
    "term_comparison",
    f"First {elapsed} days of term, {comparison_term.label} vs {current_term.label}",
    "Count",
)
comparison

,Fall 2025,Fall 2026
Sessions,31,41
Students,20,24
Prompts,83,107


In [7]:
# The same window, split by interface: the fall columns are where the two new
# surfaces show up, and the Discord column is the only like-for-like row.
by_surface = pd.DataFrame(
    {
        comparison_term.label: window_then.groupby("surface").size(),
        current_term.label: window_now.groupby("surface").size(),
    }
).fillna(0).astype(int)
by_surface.index = [SURFACE_LABELS.get(i, i) for i in by_surface.index]
surface_comparison_fig = charts.grouped_bar(
    by_surface,
    "term_comparison_by_surface",
    f"Sessions by interface, first {elapsed} days of term",
    "Sessions",
)
by_surface

,Fall 2025,Fall 2026
Discord,31,27
Chat assistant,0,8
Web,0,6


In [8]:
metrics.update("volume", {
    "elapsed_days": elapsed,
    "adoption": adoption_df.to_dict(orient="records"),
    "adoption_total_active": int(adoption_df["active"].sum()),
    "adoption_total_enrolled": int(adoption_df["enrolled"].fillna(0).sum()),
    "surface_split": surface_df.to_dict(orient="records"),
    "comparison": comparison.to_dict(),
    "comparison_by_surface": by_surface.to_dict(),
    "weekly_peak_week": str(weekly_wide.sum(axis=1).idxmax().date()) if len(weekly_wide) else None,
    "weekly_peak_sessions": int(weekly_wide.sum(axis=1).max()) if len(weekly_wide) else 0,
    "figures": {
        "adoption": adoption_fig.name,
        "weekly": weekly_fig.name,
        "surfaces": surface_fig.name,
        "comparison": comparison_fig.name,
        "comparison_by_surface": surface_comparison_fig.name,
    },
})
print("ok")

ok
